# AI 5102 - Exercise 5: AI Agent with Tool Use (ReAct)
### Plaksha University · Introduction to Large Language Models and Generative AI · Exercise 5

In this exercise you will turn the **single-turn** tool-using code from the Function Calling
session into a **multi-step agent** based on the **ReAct** paradigm
(*Reasoning + Acting*, Yao et al., 2023 — https://arxiv.org/abs/2210.03629).

A ReAct agent interleaves three primitives in a loop until the task is solved:

> **Thought** → reason about what to do next  
> **Action** → call a tool  
> **Observation** → read the tool's result  
> …repeat… → **Finish** with a final answer

**Learning objectives.** By the end you will be able to:
- Explain why single-turn function calling cannot solve multi-step tasks.
- Implement a ReAct control loop with a step budget (`max_steps`).
- Make the agent's reasoning trace explicit (Thought / Action / Observation).
- Give an agent several tools and have it *decompose* a task and *choose* among them.
- Identify, reproduce, and mitigate common agent **failure modes**.

**Outcomes:** CLO2, CLO3, CLO4.

**Model / cost.** We use NVIDIA `meta/llama-3.1-70b-instruct` (cheap). A ReAct loop makes several calls per task,
so keep `max_steps` small. *(Claude's tool-use API is
analogous if you prefer Anthropic — the loop logic is identical.)*



## Setup

In [ ]:
# Install dependencies (quiet)
%pip install -q openai requests

In [ ]:
import json
import requests
from getpass import getpass
from openai import OpenAI

print("Enter your NVIDIA API key:")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=getpass(),
    timeout=90,
)

MODEL = "meta/llama-3.1-70b-instruct"

# Quick connectivity check
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say 'ready' and nothing else."}
    ],
    max_tokens=5,
)

print(r.choices[0].message.content)

## Recap: the tools from the Function Calling session

An agent is only as capable as the tools it can call. We reuse two tools you already met:
a **weather** tool (real API) and a **calculator** (precise arithmetic — something LLMs are unreliable at).
They are provided here so this notebook is self-contained.

In [ ]:
# --- Tool 1: weather (Open-Meteo, no API key needed) ---
def get_current_weather(location: str) -> dict:
    """Return the current temperature (°F) for a city."""
    try:
        geo = requests.get(
            f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1",
            timeout=10,
        ).json()
        if not geo.get("results"):
            return {"error": f"Could not find location: {location}"}
        g = geo["results"][0]
        w = requests.get(
            f"https://api.open-meteo.com/v1/forecast?latitude={g['latitude']}"
            f"&longitude={g['longitude']}&current_weather=true&temperature_unit=fahrenheit",
            timeout=10,
        ).json()
        return {"location": g["name"], "temperature_f": w["current_weather"]["temperature"]}
    except Exception as e:
        return {"error": str(e)}

# --- Tool 2: calculator (safe arithmetic via AST) ---
import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
        ast.Mod: operator.mod}
def _eval(node):
    if isinstance(node, ast.Constant): return node.value
    if isinstance(node, ast.BinOp):  return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp):return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported expression")
def calculate(expression: str) -> dict:
    """Evaluate a basic arithmetic expression, e.g. '(45 - 32) * 2'."""
    try:
        return {"expression": expression, "result": _eval(ast.parse(expression, mode="eval").body)}
    except Exception as e:
        return {"error": f"Could not evaluate '{expression}': {e}"}

print(get_current_weather("Philadelphia"))
print(calculate("(72 - 41) * 2"))

In [ ]:
# OpenAI tool (function) schemas
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Get the current temperature in Fahrenheit for a city. Use for any weather/temperature question.",
        "parameters": {
            "type": "object",
            "properties": {"location": {"type": "string", "description": "City name, e.g. 'Tokyo'"}},
            "required": ["location"],
        },
    },
}
calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate an arithmetic expression precisely. Use for ALL arithmetic instead of doing it yourself.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "e.g. '(45 - 32) * 2'"}},
            "required": ["expression"],
        },
    },
}

---
## Part 1 — Why single-turn tool calling isn't enough (10 points)

Below is the **single-turn** helper from the Function Calling session. It offers tools on the *first* model
call, executes whatever tools the model asks for **once**, then makes a **final** call *with no tools*.
That means the model **cannot call a tool that depends on the result of a previous tool**.

In [ ]:
def chat_with_tools_single_turn(user_message, tools, available_functions):
    """One round of tool calls, then one final (tool-free) response. (From the Function Calling session.)"""
    # NVIDIA-hosted Llama rejects PARALLEL tool calls (400), so tell the
    # model to call at most one tool in this round.
    messages = [
        {"role": "system", "content": "If you use tools, call at most ONE tool at a time."},
        {"role": "user", "content": user_message},
    ]
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools, tool_choice="auto")
    msg = resp.choices[0].message
    if not msg.tool_calls:
        return msg.content
    messages.append(msg)
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        result = available_functions[tc.function.name](**args)
        print(f"[called {tc.function.name}({args}) -> {result}]")
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    # NOTE: no tools passed here -> the model cannot call another tool now.
    final = client.chat.completions.create(model=MODEL, messages=messages)
    return final.choices[0].message.content

In [ ]:
# A task that needs the calculator AFTER seeing the weather (a result-dependent second step).
tools = [weather_tool, calculator_tool]
fns = {"get_current_weather": get_current_weather, "calculate": calculate}

task = ("Use the tools to compute how many degrees warmer or colder it is right now in "
        "Philadelphia versus Tokyo (Fahrenheit). Use the calculator for the subtraction.")

print(chat_with_tools_single_turn(task, tools, fns))

### Exercise 1.1 — Diagnose the limitation (10 points) · **written**

Run the cell above **two or three times** and inspect what happened — the failure looks different
across runs. Sometimes the model calls one tool and then guesses the rest; sometimes it calls **no** tool
and instead emits text that merely *looks* like a tool call (e.g. trying to nest `get_current_weather`
inside a calculator expression). Whatever you observed, answer:

1. Which tool(s) did the single-turn helper actually manage to call in your runs — if any — and which did it **fail** to call? Why?
2. Where did the final subtraction happen: in the **calculator tool**, by the model "in its head", or not at all? How can you tell from the output?
3. In one sentence, state the structural reason single-turn tool use can't solve result-dependent multi-step tasks — regardless of which failure you happened to see.

*Your answer:*

>


---
## Part 2 — Implement the ReAct loop (35 points)

A ReAct agent keeps calling the model **in a loop**, each time giving it the tools *and* the growing
conversation (including previous observations), until the model returns an answer with **no tool call**.
A `max_steps` budget guarantees termination.

**Making the Thought visible.** On tool-calling turns the model's `content` is often empty, so we use a
robust trick: we **add a required `thought` field to every tool** ("one sentence on why you are calling
this tool"). Now every Action carries an explicit Thought — no fragile text parsing.

In [ ]:
def add_thought_param(tools):
    """Return copies of the tool schemas with a required 'thought' string added to each."""
    out = []
    for t in tools:
        t = json.loads(json.dumps(t))  # deep copy
        p = t["function"]["parameters"]
        p["properties"]["thought"] = {
            "type": "string",
            "description": "One sentence: why are you calling this tool right now?",
        }
        p["required"] = ["thought"] + p.get("required", [])
        out.append(t)
    return out

react_tools = add_thought_param([weather_tool, calculator_tool])
print(json.dumps(react_tools[0], indent=2))

### Exercise 2.1 — Write `run_react_agent` (35 points) · **code**

Complete the loop below — the rest of the notebook depends on your implementation.

In [ ]:
REACT_SYSTEM_PROMPT = """You are a ReAct agent. Solve the user's task by reasoning step by step and using tools.

At each step, call EXACTLY ONE tool with a 'thought' explaining why you are calling it.
Wait for the observation before deciding what to do next.

Never guess a number you can get from a tool.
Use the calculator for arithmetic.
The calculator accepts ONLY arithmetic expressions containing numbers and operators.
Never put function calls or tool names inside a calculator expression.

When you have enough information, reply with the final answer and NO tool call.
"""


def run_react_agent(task, tools, available_functions, max_steps=8, verbose=True):
    """
    Run a ReAct loop until the model gives a final answer or max_steps is hit.
    Returns {"answer", "trace", "stopped", "steps"}.
    """

    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": task}
    ]

    trace = []

    # --- BEGIN TODO -----------------------------------------------------------

    # YOUR CODE HERE (Exercise 2.1). Each iteration of the loop should:
    #   1. Call the chat API with `messages`, `tools`, tool_choice="auto".
    #   2. Append the assistant message to `messages` (the loop's memory).
    #   3. FINISH: if there are no tool_calls, return
    #      {"answer": msg.content, "trace": trace, "stopped": "finished", "steps": step}
    #   4. ACTION: otherwise take the FIRST tool call, parse its arguments,
    #      pop the 'thought', execute the matching function, record it in
    #      `trace`, and append a role="tool" message with the observation.
    raise NotImplementedError("Implement the ReAct loop")

    # --- END TODO -------------------------------------------------------------

    if verbose:
        print(f"[stopped] hit max_steps={max_steps} without finishing")

    return {
        "answer": None,
        "trace": trace,
        "stopped": "max_steps",
        "steps": max_steps
    }

---
## Part 3 — Run the agent on a multi-step task (20 points)

In [ ]:
# The same task that defeated the single-turn helper — now solved with chaining.
result = run_react_agent(task, react_tools, fns, max_steps=8)
print("\n=== ANSWER ===")
print(result["answer"])
print(f"\n(steps: {result['steps']}, stopped: {result['stopped']}, tool calls: {len(result['trace'])})")

### Exercise 3.1 — Your own multi-step task (20 points) · **code + written**

Write a task that **requires at least 3 steps and both tools**, where a later action depends on an
earlier observation. Run your agent on it and paste the trace.

Then, in 2–3 sentences: how did the agent **decompose** the task? Did it pick the right tools in the
right order?

In [ ]:
# STUDENT TODO: replace with your own multi-step task
my_task = "REPLACE ME with a task that needs >=3 steps and both tools."
# run_react_agent(my_task, react_tools, fns, max_steps=8)

*Your analysis:*

>


---
## Part 4 — Failure-mode analysis (25 points)

Real agents fail in characteristic ways. Reproduce and mitigate at least **three**.

In [ ]:
# Failure demo A: non-termination -> the step budget saves you.
# A deliberately under-budgeted / open-ended task hits max_steps.
loopy = run_react_agent(
    "Keep checking the weather in a different world capital until you find one hotter than 200F, "
    "then report it.", react_tools, fns, max_steps=4)
print("\nstopped:", loopy["stopped"], "| answer:", loopy["answer"])

In [ ]:
# Failure demo B: bad tool input -> does the agent recover from an error observation?
recover = run_react_agent(
    "What is the temperature in the city of Xyzzyxq? If that city is not found, "
    "report the temperature in Delhi instead.", react_tools, fns, max_steps=6)
print("\nanswer:", recover["answer"])

### Exercise 4.1 — Document three failure modes (25 points) · **written + code**

For **each** of three failure modes, give: (a) a one-line description, (b) a concrete input that triggers
it (show the trace), and (c) a mitigation. Choose from — or go beyond — this list:

| Failure mode | Typical cause | Example mitigation |
|---|---|---|
| Non-termination / hits `max_steps` | no stopping condition met | step budget; better system prompt |
| Hallucinated tool arguments | model invents inputs | schema validation; require a `thought` |
| Wrong tool / over-calling | poor tool descriptions | sharper descriptions; cost cap |
| Premature stop (guesses) | model answers without acting | prompt: "never guess a computable value" |
| No error recovery | ignores an error observation | instruct re-planning on `error` |
| Cost/latency blow-up | steps × tokens grow | measure tokens vs. steps; cap steps |

*Your analysis (add code cells for the triggering traces):*

>


---
## Part 5 — Reflection (10 points) · **written**

1. **ReAct vs. plain function calling.** What exactly does the *loop* add over the single-turn version from Part 1?
2. **ReAct vs. chain-of-thought.** Both surface reasoning. What can ReAct do that pure CoT prompting cannot? *(Hint: the environment.)*
3. **When NOT to use an agent.** Give one task where a single call is better, considering cost, latency, and reliability.
4. Skim Yao et al. (2023), *ReAct: Synergizing Reasoning and Acting in Language Models* (https://arxiv.org/abs/2210.03629). Name one benefit they report from interleaving reasoning and acting.

*Your answers:*

>


---
## Grading rubric (100 points)

| Part | Points | Criteria |
|---|---|---|
| 1 — Diagnose single-turn | 10 | Correctly explains why single-turn can't chain result-dependent steps |
| 2 — `run_react_agent` | 35 | Correct loop; accumulates history; Thought/Action/Observation; `max_steps` guard |
| 3 — Multi-step task | 20 | Working agent on a ≥3-step, 2-tool task; sound decomposition analysis |
| 4 — Failure modes | 25 | Three modes reproduced with traces + mitigations |
| 5 — Reflection | 10 | Thoughtful; connects to CoT and the ReAct paper |
| **Total** | **100** | |

### Submission checklist
- [ ] All cells run top-to-bottom without errors, outputs visible
- [ ] `run_react_agent` implemented and returns the structured trace
- [ ] Part 3 shows a full Thought→Action→Observation→Answer trace
- [ ] Three failure modes documented with triggering traces and mitigations
- [ ] Reflection answered

---
### Appendix (optional enrichment) — classic text-based ReAct
The original paper used **no** structured tool API: the model emitted `Thought: … / Action: tool[input]`
as text, the harness parsed it, ran the tool, and appended `Observation: …`. Re-implement the loop that
way (regex-parsing `Action:` lines) to appreciate why structured function calling replaced it.
